# Conciliacion de stock de Casa Tahona

Corte del 12/10/2026. Reconstruccion independiente de los 678 saldos usando los archivos de evidencia adjuntos. No se conecta a la base de datos ni modifica el sistema. Los datos representan unidades base y el estado visible en el momento de extraccion. El producto piloto incluye devoluciones; los demas reproducen el reporte actual. No sustituye un conteo fisico.

In [1]:
import json
from pathlib import Path
from datetime import datetime, date
from zoneinfo import ZoneInfo
import re

root = Path.cwd()
if not (root / 'evidencia_consultada.json').exists():
    root = root / 'outputs/auditoria-casa-tahona-2026-10-12'
raw = json.loads((root / 'evidencia_consultada.json').read_text())
analysis = json.loads((root / 'resultado_auditoria.json').read_text())
zone = ZoneInfo('America/Mexico_City')
pilot = '5285790a-0dff-4c16-a268-9a401ad03336'
def timestamp(value):
    return datetime.fromisoformat(value.replace('Z', '+00:00'))
def purchase_time(p):
    created = timestamp(p['created_at']).astimezone(zone)
    day = date.fromisoformat(p['purchase_date'])
    return created.replace(year=day.year, month=day.month, day=day.day)
def qty(row):
    return float(row['qty_base'] if row['qty_base'] is not None else row['qty'])

checked = 0
for product in analysis['rows']:
    pid = product['id']
    events = []
    for row in raw['purchases']:
        parent = row['purchase']
        if row['product_id'] == pid and parent['branch_id'] == 1 and not parent['deleted_at']:
            events.append((purchase_time(parent), 'movement', qty(row)))
    for row in raw['sales']:
        parent = row['sale']
        if row['product_id'] != pid or parent['branch_id'] != 1:
            continue
        if not parent['deleted_at'] or pid == pilot:
            events.append((timestamp(parent['created_at']), 'movement', -qty(row)))
        if parent['deleted_at'] and pid == pilot:
            events.append((timestamp(parent['deleted_at']), 'movement', qty(row)))
    audits = sorted((a for a in raw['productAudit'] if a['entity_id'] == pid),
                    key=lambda a: timestamp(a['timestamp']), reverse=True)[:50]
    for a in audits:
        if a['action_type'] not in ('CREAR', 'ACTUALIZAR'):
            continue
        match = re.search(r'stock ajustado manualmente.*de\s+([\d.]+)\s+a\s+([\d.]+)', a['description'], re.I)
        before = (a['previous_data'] or {}).get('stock')
        after = (a['new_data'] or {}).get('stock')
        if match:
            before = before if before is not None else float(match.group(1))
            after = after if after is not None else float(match.group(2))
        if isinstance(before, (int,float)) and isinstance(after, (int,float)):
            events.append((timestamp(a['timestamp']), 'snapshot', after))
    balance = 0
    for when, kind, amount in sorted(events, key=lambda event: event[0]):
        balance = amount if kind == 'snapshot' else balance + amount
    assert abs(balance - product['report']) < 0.000001, (product['name'], balance, product['report'])
    stored = sum(float(s['qty']) for s in raw['stocks'] if s['product_id'] == pid and s['branch_id'] == 1)
    assert abs(stored - product['stock']) < 0.000001
    checked += 1

active = [r for r in analysis['rows'] if r['active']]
differences = [r for r in active if r['delta'] != 0]
print(f'{checked} productos verificados con una reconstruccion independiente.')
print(f'{len(active)} activos; {len(differences)} diferencias adicionales; {len(active)-len(differences)} saldos iguales.')
assert raw['manifest']['stockStable'] is True


678 productos verificados con una reconstruccion independiente.
637 activos; 11 diferencias adicionales; 626 saldos iguales.
